# 🏰 천상계 2차시 · 물의 도시 — 호와 부채꼴, 중심각의 비밀

**원, 더 월드 · 천상계(Manim 영상 제작 실습)** | 이름: ____________________ `v1 (2026-09-21)`

🗺️ **연결된 도시**: CITY 02 물의 도시 — 원과 부채꼴, 호의 길이 (게임: 빗방울 요격)

📐 **오늘의 수학**: 호의 길이와 부채꼴의 넓이는 **중심각의 크기에 정비례**한다.

🎯 **학습 목표**

- 원 위의 두 점으로 호·현·부채꼴을 코드로 그릴 수 있다.
- `ValueTracker`로 중심각을 움직이며 호의 길이가 변하는 모습을 만들 수 있다.
- 중심각이 2배가 되면 호의 길이도 2배가 됨을 영상으로 보일 수 있다.

> 🌌 물의 도시에서는 중심각을 맞춰 빗방울을 요격했죠. 그 뒤에 숨은 '정비례'를 영상으로 만들어요.

In [ ]:
#@title ⚙️ 1단계: Manim 설치 (재생 ▶ 누르고 약 2분 대기)
# ⚠️ 이 셀은 고치지 말고 그대로 실행하세요. 빨간 경고가 나와도 괜찮아요.
!sudo apt update -qq
!sudo apt install -y -qq libcairo2-dev libpango1.0-dev ffmpeg fonts-nanum > /dev/null
!fc-cache -f > /dev/null
!pip install -q manim
print("📦 설치 끝!")
print("👉 이제 메뉴 [런타임] → [세션 다시 시작] 을 누른 뒤,")
print("   아래 '2단계' 셀부터 실행하세요. (이 셀은 다시 실행하지 않아도 돼요)")

> ### 🔄 잠깐! 설치가 끝나면 꼭 **[런타임] → [세션 다시 시작]** 을 누르세요.
> 새로 설치된 도구를 Colab이 제대로 불러오려면 한 번 재시작해야 해요.
> 재시작 후에는 위의 설치 셀 말고 **아래 셀부터** 실행하면 됩니다.

In [ ]:
#@title ✅ 2단계: 준비 확인 (세션 다시 시작 후 실행)
from manim import *
import numpy as np
print("✅ 준비 완료! 이제 아래 셀들을 실행할 수 있어요.")

## 1. 호, 현, 부채꼴 — 원의 부분에 이름 붙이기

원 위의 두 점 A, B를 잡으면 원이 두 부분으로 나뉘어요.

| 이름 | 뜻 | Manim 코드 |
|---|---|---|
| 호 AB | 원 위 두 점 사이의 **굽은 부분** | `Arc(radius=r, start_angle=…, angle=…)` |
| 현 AB | 두 점을 잇는 **선분** | `Line(A, B)` |
| 부채꼴 | 두 반지름과 호로 둘러싸인 부분 | `Sector(radius=r, angle=…)` |
| 중심각 | 부채꼴의 두 반지름이 이루는 각 ∠AOB | `Angle(선1, 선2)` |

> 💡 원 위의 점은 `r * np.array([np.cos(θ), np.sin(θ), 0])`로 구해요. `θ`는 `60 * DEGREES`처럼 써요.

In [ ]:
%%manim -qm -v WARNING ArcChordSector

class ArcChordSector(Scene):
    def construct(self):
        r = 2.5
        O = ORIGIN
        A = r * np.array([np.cos(20 * DEGREES), np.sin(20 * DEGREES), 0])
        B = r * np.array([np.cos(110 * DEGREES), np.sin(110 * DEGREES), 0])

        circle = Circle(radius=r, color=GREY)
        dots = VGroup(Dot(O, color=ORANGE), Dot(A, color=YELLOW), Dot(B, color=YELLOW))
        names = VGroup(Text("O", font_size=28).next_to(O, DOWN + LEFT, buff=0.1),
                       Text("A", font_size=28, color=YELLOW).next_to(A, RIGHT, buff=0.1),
                       Text("B", font_size=28, color=YELLOW).next_to(B, UP, buff=0.1))
        self.play(Create(circle), FadeIn(dots), Write(names))

        # 호 AB (노랑)
        arc = Arc(radius=r, start_angle=20 * DEGREES, angle=90 * DEGREES, color=YELLOW, stroke_width=8)
        t_arc = Text("호 AB", font="NanumGothic", font_size=28, color=YELLOW).move_to(1.35 * (A + B) / 2 * 1.15)
        self.play(Create(arc), Write(t_arc)); self.wait(0.5)

        # 현 AB (초록)
        chord = Line(A, B, color=GREEN)
        t_chord = Text("현 AB", font="NanumGothic", font_size=28, color=GREEN).next_to(chord.get_center(), DOWN + LEFT, buff=0.15)
        self.play(Create(chord), Write(t_chord)); self.wait(0.5)

        # 부채꼴 AOB (파랑) + 중심각
        sector = Sector(radius=r, start_angle=20 * DEGREES, angle=90 * DEGREES, color=BLUE, fill_opacity=0.35, stroke_width=0)
        OA, OB = Line(O, A), Line(O, B)
        ang = Angle(OA, OB, radius=0.6, color=ORANGE)
        t_ang = Text("중심각 90°", font="NanumGothic", font_size=28, color=ORANGE).to_corner(UR)
        t_sec = Text("부채꼴 AOB", font="NanumGothic", font_size=28, color=BLUE).to_corner(DL)
        self.play(FadeIn(sector), Create(OA), Create(OB), Create(ang), Write(t_ang), Write(t_sec))
        self.wait(2)

## 2. ⭐ 말없는 증명 — 중심각이 커지면 호의 길이는?

`ValueTracker`는 **변하는 수**를 담는 통이에요. 중심각을 0°에서 270°까지 키우면서
호의 길이와 넓이가 같이 커지는 것을 숫자판으로 확인해요.

$$\text{호의 길이}=2\pi r\times\frac{\text{중심각}}{360^\circ}, \qquad \text{넓이}=\pi r^2\times\frac{\text{중심각}}{360^\circ}$$

`always_redraw(함수)`는 값이 바뀔 때마다 도형을 다시 그려 줘요.

In [ ]:
%%manim -qm -v WARNING SectorGrow

class SectorGrow(Scene):
    def construct(self):
        r = 2.4                                    # 값이 π의 깔끔한 배수로 나오는 반지름
        theta = ValueTracker(1)                    # 중심각(도)
        shift = LEFT * 2.8
        circle = Circle(radius=r, color=GREY).shift(shift)
        center = Dot(color=ORANGE).shift(shift)

        sector = always_redraw(lambda: Sector(radius=r, start_angle=0, angle=theta.get_value() * DEGREES,
                                              color=BLUE, fill_opacity=0.5, stroke_width=0).shift(shift))
        arc = always_redraw(lambda: Arc(radius=r, start_angle=0, angle=theta.get_value() * DEGREES,
                                        color=YELLOW, stroke_width=8).shift(shift))

        def readout():
            t = theta.get_value()
            arc_len = 2 * r * t / 360                 # 호의 길이 = (이 값) × π
            area = r * r * t / 360                    # 넓이 = (이 값) × π
            txt = Text(f"중심각  {t:5.0f}°\n호의 길이  {arc_len:4.2f}π\n넓이  {area:4.2f}π",
                       font="NanumGothic", font_size=30, line_spacing=1.2)
            return txt.move_to(RIGHT * 3.2)
        panel = always_redraw(readout)

        title = Text("중심각과 호의 길이 사이의 관계", font="NanumGothic", font_size=36).to_edge(UP)
        self.play(Write(title))
        self.play(Create(circle), FadeIn(center))
        self.add(sector, arc, panel)
        self.play(theta.animate.set_value(90), run_time=2)
        self.wait(0.5)
        self.play(theta.animate.set_value(180), run_time=2)   # 2배 → 호도 2배!
        self.wait(0.5)
        self.play(theta.animate.set_value(270), run_time=2)
        self.wait(1.5)

## 3. 중심각 2배 = 호의 길이 2배 — 나란히 비교

같은 원에서 중심각 45°와 90°인 부채꼴을 나란히 놓고, 호를 **펴서** 길이를 비교해 봐요.
호를 곧게 펴면 막대가 되는데, 90°짜리 막대는 45°짜리의 정확히 2배예요.

In [ ]:
%%manim -qm -v WARNING DoubleAngleDoubleArc

class DoubleAngleDoubleArc(Scene):
    def construct(self):
        r = 2
        cases = [(45, LEFT * 3.2, YELLOW), (90, RIGHT * 3.2, GREEN)]
        bars = VGroup()
        for deg, pos, color in cases:
            circle = Circle(radius=r, color=GREY).move_to(pos)
            sector = Sector(radius=r, start_angle=0, angle=deg * DEGREES, color=color, fill_opacity=0.4, stroke_width=0).shift(pos)
            arc = Arc(radius=r, start_angle=0, angle=deg * DEGREES, color=color, stroke_width=8).shift(pos)
            label = Text(f"중심각 {deg}°", font="NanumGothic", font_size=28, color=color).next_to(circle, UP)
            self.play(Create(circle), FadeIn(sector), Create(arc), Write(label), run_time=1)

            # 호를 펴서 막대로 (길이 = 호의 길이)
            length = 2 * PI * r * deg / 360
            bar = Line(ORIGIN, RIGHT * length, color=color, stroke_width=8).next_to(circle, DOWN, buff=0.4)
            self.play(Transform(arc.copy(), bar), run_time=1.2)
            bars.add(bar)
            self.add(bar)

        t1 = Text(f"{2*PI*r*45/360:.2f}", font_size=26, color=YELLOW).next_to(bars[0], DOWN)
        t2 = Text(f"{2*PI*r*90/360:.2f}", font_size=26, color=GREEN).next_to(bars[1], DOWN)
        msg = Text("중심각 2배 → 호의 길이도 2배", font="NanumGothic", font_size=32).to_edge(DOWN)
        self.play(Write(t1), Write(t2))
        self.play(Write(msg))
        self.wait(2)

## 🏆 도전 과제 — 빗방울 요격! 중심각 맞추기

물의 도시 게임처럼, **목표 호**(빨강)의 길이와 같아지도록 **내 부채꼴**(파랑)의 중심각을 맞춰 보세요.

1. 목표 호의 중심각은 `target = 120`. 내 부채꼴의 `my_angle`을 바꿔 두 호의 길이가 같아지게 하세요.
2. 반지름이 다르면? `r_target = 2, r_me = 1` 로 바꾸고, 길이가 같아지려면 `my_angle`이 몇이어야 하는지 계산해 보세요. (힌트: 호의 길이 = 2πr × 중심각/360)
3. (보너스) `ValueTracker`를 써서 내 부채꼴이 0°부터 자라다가 목표에서 멈추게 해 보세요.

In [ ]:
%%manim -qm -v WARNING RaindropIntercept

class RaindropIntercept(Scene):
    def construct(self):
        r_target, r_me = 2, 2          # 2번 과제: r_me = 1 로 바꿔 보세요
        target = 120                   # 목표 호의 중심각
        my_angle = 60                  # 1번 과제: 이 값을 바꿔서 길이를 맞추세요

        left, right = LEFT * 3.2, RIGHT * 3.2
        c1 = Circle(radius=r_target, color=GREY).move_to(left)
        c2 = Circle(radius=r_me, color=GREY).move_to(right)
        arc_t = Arc(radius=r_target, start_angle=0, angle=target * DEGREES, color=RED, stroke_width=8).shift(left)
        arc_m = Arc(radius=r_me, start_angle=0, angle=my_angle * DEGREES, color=BLUE, stroke_width=8).shift(right)
        self.play(Create(c1), Create(c2))
        self.play(Create(arc_t), Create(arc_m))

        len_t = 2 * PI * r_target * target / 360
        len_m = 2 * PI * r_me * my_angle / 360
        t1 = Text(f"목표 호 {len_t:.2f}", font="NanumGothic", font_size=28, color=RED).next_to(c1, DOWN)
        t2 = Text(f"내 호 {len_m:.2f}", font="NanumGothic", font_size=28, color=BLUE).next_to(c2, DOWN)
        self.play(Write(t1), Write(t2))

        ok = abs(len_t - len_m) < 0.01
        msg = Text("🎯 요격 성공!" if ok else "다시 조준!", font="NanumGothic", font_size=36,
                   color=GREEN if ok else ORANGE).to_edge(UP)
        self.play(Write(msg))
        self.wait(2)

## 📝 오늘 배운 것 정리

- 호(`Arc`), 현(`Line`), 부채꼴(`Sector`), 중심각(`Angle`)을 코드로 그린다.
- 호의 길이 = 2πr × (중심각/360), 부채꼴의 넓이 = πr² × (중심각/360).
- **호의 길이와 넓이는 중심각에 정비례** → 중심각 2배면 둘 다 2배.
- `ValueTracker` + `always_redraw`로 움직이는 도형과 숫자판을 만든다.

**다음 시간 (얼음의 도시)**: 삼각형의 세 꼭짓점을 지나는 원, **외접원과 외심**을 작도해요.